In [269]:
from constants import users_list, data_path
from lib import spoti, genre_normalizer, plotting, preprocessing, dimensionality_reduction

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import json
import os
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.nn import functional as F
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import mean_squared_error
import random
import time
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors, NeighborhoodComponentsAnalysis
from sklearn.metrics import pairwise_distances
import pickle
import numpy as np
from typing import List, Dict, Tuple, Optional, Any, Callable
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from IPython.display import display, HTML
from recommenders.matrix_dataset import MatrixDataset
from recommenders.lmf import LogisticMatrixFactorization

In [270]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32

if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    device = mps_device
    x = torch.ones(1, device=mps_device)
    print(x)
else:
    print ("MPS device not found.")

if device == torch.device("cuda"):
    dtype = torch.float32
    print("Using CUDA.")
elif device == torch.device("cpu"):
    dtype = torch.float64
    print("Using CPU.")
elif device == torch.device("mps"):
    dtype = torch.float32
    print("Using MPS.")

# device = "cpu"
# dtype = torch.float32

tensor([1.], device='mps:0')
Using MPS.


# Import data

In [271]:
df = spoti.load_all_tracks(
    base_path=data_path.DATA_PATH,
    users=users_list.USERS,
    load_spotify_tracks=False,
    penality_factors={"short_term": 1, "medium_term": 1, "long_term": 1},
)
df

,album,artists,available_markets,disc_number,duration_ms,explicit,external_ids,external_urls,href,id,...,time_range,affinity,username,release_year,normalized_genres,added_at,episode,track,added_by,playlist_id
0,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,196426,False,{'isrc': 'USSM12301260'},{'spotify': 'https://open.spotify.com/track/75...,https://api.spotify.com/v1/tracks/75rqqKvzJCGv...,75rqqKvzJCGv2oq9C4yFDt,...,medium_term,1.00,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
1,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,137533,True,{'isrc': 'USSM12109218'},{'spotify': 'https://open.spotify.com/track/2F...,https://api.spotify.com/v1/tracks/2FYGZDfsAnNs...,2FYGZDfsAnNsrm1gVbyKnG,...,medium_term,0.98,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
2,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,162906,True,{'isrc': 'USSM12109222'},{'spotify': 'https://open.spotify.com/track/4k...,https://api.spotify.com/v1/tracks/4kroNlz8BTfs...,4kroNlz8BTfswE4M0i3YCh,...,medium_term,0.96,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
3,"{'album_type': 'ALBUM', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,89749,False,{'isrc': 'USSM12208854'},{'spotify': 'https://open.spotify.com/track/2N...,https://api.spotify.com/v1/tracks/2N3YZ075lq9z...,2N3YZ075lq9z1ObaAiX6l1,...,medium_term,0.94,jaslkh,2022,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
4,"{'album_type': 'SINGLE', 'artists': [{'externa...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AE, AR, AT, AU, BE, BG, BH, BO, BR, CA, C...",1,174044,False,{'isrc': 'USSM12300114'},{'spotify': 'https://open.spotify.com/track/2S...,https://api.spotify.com/v1/tracks/2SiAcexM2p1y...,2SiAcexM2p1yX6joESbehd,...,medium_term,0.92,jaslkh,2023,"[spanish, pop, rnb]",NaT,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12424,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AD, AL, AM, AT, AZ, BA, BE, BG, BY, CH, CW, C...",1,251880,False,{'isrc': 'GBN9Y1100001'},{'spotify': 'https://open.spotify.com/track/3z...,https://api.spotify.com/v1/tracks/3z7dWKRsjDNM...,3z7dWKRsjDNM24ohLKZBnA,...,NaN,NaN,dany,1967,"[rock, rock, rock, rock, rock, rock, rock]",2022-12-30 08:42:31+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12425,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,193853,False,{'isrc': 'GBLTP1700005'},{'spotify': 'https://open.spotify.com/track/1V...,https://api.spotify.com/v1/tracks/1VofMhhL98pe...,1VofMhhL98pewltVGBSmCW,...,NaN,NaN,dany,2017,"[rock, rock, rock, rock, rock]",2020-08-10 14:53:07+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12426,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,295493,False,{'isrc': 'GBLTP1700011'},{'spotify': 'https://open.spotify.com/track/0K...,https://api.spotify.com/v1/tracks/0KE7apgczHNY...,0KE7apgczHNYiXIvMUY0Fc,...,NaN,NaN,dany,2017,"[rock, rock, rock, rock, rock]",2020-08-10 14:53:13+00:00,False,True,ssscnm71jg9pv7fryho4d5qgs,674vFKUXNUjlUxXhAnQvG9
12427,"{'album_type': 'album', 'artists': [{'external...",[{'external_urls': {'spotify': 'https://open.s...,"[AR, AU, AT, BE, BO, BR, BG, CA, CL, CO, CR, C...",1,467306,False,{'isrc': 'GBLTP1700014'},{'spotify': 'https://open.spotify.com/track/6v...,https://api.spotify.com/v1/tracks/6vbRA9yAAgIX...,6vbRA9yAAgIXtDlmhyNqPq,...,NaN,N

# Logistic Matrix Factorization for Implicit Feedback Data (Logistic MF)
From [Christopher C. Johnson - Logistic Matrix Factorization for Implicit Feedback Data](https://web.stanford.edu/~rezab/nips2014workshop/submits/logmat.pdf)

## Setup the dataset

In [272]:
df_matrix_mf = df.copy()
df_matrix_mf.loc[df_matrix_mf["type"] == "liked_track", "affinity"] = 0.5
df_matrix_mf.loc[df_matrix_mf["type"] == "playlist", "affinity"] = 0.3

used_types = ["top_track", "liked_track", "playlist"]
used_types = ["top_track"]
df_matrix_mf = df_matrix_mf[df["type"].isin(used_types)]
df_matrix_mf[["username", "id", "affinity"] + spoti.NUMERICAL_FEATURES]
df_matrix_mf["affinity"] *= 100

In [273]:
matrix_mf = MatrixDataset(df_matrix_mf, "username", "id", "affinity", device=device, dtype=dtype)
R = matrix_mf.R
R

tensor([[ 0.,  0., 62.,  ...,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  ...,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  ...,  0.,  0., 84.],
        ...,
        [ 0.,  0.,  0.,  ...,  0.,  0.,  0.],
        [44., 60.,  0.,  ...,  0.,  0.,  0.],
        [ 0.,  0.,  0.,  ..., 92.,  0.,  0.]], device='mps:0')

In [274]:
R.shape

torch.Size([8, 923])

In [275]:
alpha = matrix_mf.compute_alpha().item()
R *= alpha
alpha

0.10527777671813965

In [276]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import List, Optional


class MultiVAE(nn.Module):
    """
    Variational Autoencoder with Multinomial Likelihood (Multi-VAE).

    This VAE is designed for collaborative filtering tasks. The architecture
    comprises of encoder (q-network) and decoder (p-network) layers. The encoder
    produces a latent representation, and the decoder reconstructs the input data.

    Attributes:
        encoder_dims (List[int]): Dimensions for the encoder layers.
        decoder_dims (List[int]): Dimensions for the decoder layers.
        dropout_rate (float): Dropout rate for regularization.

    References:
        [1] Liang, Dawen, et al. "Variational autoencoders for collaborative filtering."
            Proceedings of the 2018 World Wide Web Conference. 2018.
        [2] Variational autoencoders for collaborative filtering
            by @dawenl.
            https://github.com/dawenl/vae_cf
        [3] Variational Autoencoders for Collaborative Filtering - Implementation in PyTorch
            by @younggyoseo.
            https://github.com/younggyoseo/vae-cf-pytorch
    """

    def __init__(
        self,
        encoder_dims: List[int],
        decoder_dims: Optional[List[int]] = None,
        dropout_rate: float = 0.5,
    ) -> None:
        super(MultiVAE, self).__init__()
        self.encoder_dims = encoder_dims
        self.decoder_dims = decoder_dims if decoder_dims else encoder_dims[::-1]

        assert (
            self.decoder_dims[0] == encoder_dims[-1]
        ), "Output dimension of encoder must match input dimension of decoder."
        assert (
            self.decoder_dims[-1] == encoder_dims[0]
        ), "Latent dimension mismatch between encoder and decoder."

        # Modify the last dimension of encoder for mean and variance
        modified_encoder_dims = self.encoder_dims[:-1] + [self.encoder_dims[-1] * 2]
        self.encoder_layers = nn.ModuleList(
            [
                nn.Linear(in_features, out_features)
                for in_features, out_features in zip(
                    modified_encoder_dims[:-1], modified_encoder_dims[1:]
                )
            ]
        )
        self.decoder_layers = nn.ModuleList(
            [
                nn.Linear(in_features, out_features)
                for in_features, out_features in zip(
                    self.decoder_dims[:-1], self.decoder_dims[1:]
                )
            ]
        )

        self.dropout = nn.Dropout(dropout_rate)
        self.initialize_weights()

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        mu, logvar = self.encode(input)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

    def encode(self, input: torch.Tensor) -> torch.Tensor:
        h = F.normalize(input)
        h = self.dropout(h)

        for i, layer in enumerate(self.encoder_layers):
            h = layer(h)
            if i != len(self.encoder_layers) - 1:
                h = torch.tanh(h)
            else:
                mu = h[:, : self.encoder_dims[-1]]
                logvar = h[:, self.encoder_dims[-1] :]
        return mu, logvar

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return eps.mul(std).add_(mu)
        else:
            return mu

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        h = z
        for i, layer in enumerate(self.decoder_layers):
            h = layer(h)
            if i != len(self.decoder_layers) - 1:
                h = torch.tanh(h)
        return h

    def initialize_weights(self) -> None:
        for layer in self.encoder_layers + self.decoder_layers:
            size = layer.weight.size()
            fan_out, fan_in = size[0], size[1]
            std = np.sqrt(2.0 / (fan_in + fan_out))
            layer.weight.data.normal_(0.0, std)
            layer.bias.data.normal_(0.0, 0.001)

    def loss(
        self,
        recon_x: torch.Tensor,
        x: torch.Tensor,
        mu: torch.Tensor,
        logvar: torch.Tensor,
        anneal: float = 1.0,
    ) -> torch.Tensor:
        """
        Loss function for MultiVAE.

        Combines Binary Cross Entropy (BCE) and Kullback–Leibler Divergence (KLD)
        to form the Variational Autoencoder loss.

        Parameters:
            recon_x (torch.Tensor): Reconstructed input.
            x (torch.Tensor): Original input.
            mu (torch.Tensor): Mean from the latent space.
            logvar (torch.Tensor): Log variance from the latent space.
            anneal (float): Annealing factor for KLD.

        Returns:
            torch.Tensor: Calculated loss.
        """
        BCE = -torch.mean(torch.sum(F.log_softmax(recon_x, 1) * x, -1))
        KLD = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
        return BCE + anneal * KLD

In [277]:
users, items = R.shape

In [278]:
batch_size = 32

# train_set, val_set = train_test_split(R, test_size=0.2, random_state=42)

# # Convert the train and validation sets into TensorDataset objects
# train_dataset = TensorDataset(train_set)
# val_dataset = TensorDataset(val_set)

# # Create DataLoaders for batching
# train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

train_set = R
train_dataset = TensorDataset(train_set)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)

In [279]:
# Define dimensions for the encoder and decoder
encoder_dims = [R.shape[1], 3]
decoder_dims = encoder_dims[::-1]  # Symmetric decoder

# Initialize the model
model = MultiVAE(encoder_dims, decoder_dims).to(device)

In [280]:
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training loop
num_epochs = 1000  # Number of epochs
val_losses = []  # Keep track of the validation loss
train_losses = []  # Keep track of the training loss
for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    for data in train_loader:
        data = data[0]
        # Assuming data is a batch of user-item interactions
        optimizer.zero_grad()
        recon_batch, mu, logvar = model(data)
        loss = model.loss(recon_batch, data, mu, logvar)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)
    train_losses.append(train_loss)

    # val_loss = 0.0
    # model.eval()
    # with torch.no_grad():
    #     for data in val_loader:
    #         data = data[0]
    #         recon_batch, mu, logvar = model(data)
    #         loss = loss_function(recon_batch, data, mu, logvar)
    #         val_loss += loss.item()
    #     val_loss /= len(val_loader)
    #     val_losses.append(val_loss)

    print(
        f"Epoch {epoch + 1}/{num_epochs}: Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}"
    )

Epoch 1/1000: Train Loss: 5502.6851, Val Loss: 16293.4531
Epoch 2/1000: Train Loss: 5489.7290, Val Loss: 16293.4531
Epoch 3/1000: Train Loss: 5487.2002, Val Loss: 16293.4531
Epoch 4/1000: Train Loss: 5491.2246, Val Loss: 16293.4531
Epoch 5/1000: Train Loss: 5479.0884, Val Loss: 16293.4531
Epoch 6/1000: Train Loss: 5509.8271, Val Loss: 16293.4531
Epoch 7/1000: Train Loss: 5455.9365, Val Loss: 16293.4531
Epoch 8/1000: Train Loss: 5468.4092, Val Loss: 16293.4531
Epoch 9/1000: Train Loss: 5448.8560, Val Loss: 16293.4531
Epoch 10/1000: Train Loss: 5450.2690, Val Loss: 16293.4531
Epoch 11/1000: Train Loss: 5437.5947, Val Loss: 16293.4531
Epoch 12/1000: Train Loss: 5440.6343, Val Loss: 16293.4531
Epoch 13/1000: Train Loss: 5388.4004, Val Loss: 16293.4531
Epoch 14/1000: Train Loss: 5398.8203, Val Loss: 16293.4531
Epoch 15/1000: Train Loss: 5412.3833, Val Loss: 16293.4531
Epoch 16/1000: Train Loss: 5426.7051, Val Loss: 16293.4531
Epoch 17/1000: Train Loss: 5415.8701, Val Loss: 16293.4531
Epoch 

In [281]:
go.Figure(
    data=[
        go.Scatter(
            x=list(range(len(train_losses))),
            y=train_losses,
            name="Train Loss",
            mode="lines",
        ),
        go.Scatter(
            x=list(range(len(val_losses))),
            y=val_losses,
            name="Validation Loss",
            mode="lines",
        ),
    ],
    layout=go.Layout(
        title="Training and Validation Loss",
        xaxis=dict(title="Epoch"),
        yaxis=dict(title="Loss"),
    ),
)

In [282]:
# Get all users latent representations
model.eval()
with torch.no_grad():
    mu, _ = model.encode(R)
    z = model.reparameterize(mu, torch.zeros_like(mu))
    z = z.cpu().numpy()

df_matrix_mf_latent = df_matrix_mf.copy()

user_latent_columns = [f"user_latent_{i}" for i in range(encoder_dims[-1])]
for username, user_df in df_matrix_mf_latent.groupby("username"):
    index = matrix_mf.usernames_to_ids([username])[0]
    latent_representation = z[index]
    df_matrix_mf_latent.loc[
        df_matrix_mf_latent["username"] == username, user_latent_columns
    ] = latent_representation

plotting.plot_latent_space(
    df_matrix_mf_latent,
    color=df_matrix_mf_latent["username"],
    text=df_matrix_mf_latent["username"],
    latent_columns=user_latent_columns,
    title="User Latent Space",
).show()